<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Data_Validation_Schema_Checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 3.3 — Data Validation & Schema Checks

**Module 1: Programming & Data Foundations**  
**Week 3 | Session 3.3**

---

## What we will cover today

1. What is "dirty" data and why it matters
2. Handling missing values
3. Fixing wrong data types
4. Identifying inconsistent formats
5. Schema validation basics

---

> **The goal of this session:** By the end, you should be able to take a messy, real-world dataset and run a set of checks that tell you exactly what is wrong with it — before it causes problems downstream.

---
## Setup — Run this first

This cell imports everything we need for today. Run it before anything else.

In [ ]:
import csv
import json
from io import StringIO

print("Ready to go!")

Ready to go!


---
## Part 1 — What is "Dirty" Data?

### The analogy

Imagine you work at a hospital and someone hands you a stack of patient intake forms. Some forms are filled in neatly. But others have problems:

- A patient's age is written as `"twenty-five"` instead of `25`
- The date of birth column is blank for three patients
- One form says the blood pressure is `"normal"` while all others have numbers like `120/80`
- Two patients are listed as `"Male"` and one as `"male"` and one as `"M"`

If you try to compute the average age of patients, or sort by date, or filter by blood pressure — these problems will cause your code to either **crash** or give you **wrong answers**. Both outcomes are bad.

That is dirty data.

---

### The four types of dirty data we will fix today

| Problem | Example |
|---|---|
| Missing values | A cell is empty or blank |
| Wrong data types | Age stored as `"25"` (text) instead of `25` (number) |
| Inconsistent formats | Dates as `"01/05/2024"` vs `"2024-01-05"` |
| Schema violations | A required field is absent, or a value is out of allowed range |

---
## Our Dataset for Today

We will work with a dataset of **employee records** from a fictional company. This data was collected from multiple sources — some entered manually, some from old HR systems — so it has a variety of real-world problems.

We will use this same dataset throughout the entire session and fix it step by step.

Here is what each column is supposed to contain:

| Column | Expected Type | Notes |
|---|---|---|
| `employee_id` | Integer | Unique, required |
| `name` | String | Required |
| `age` | Integer | Must be between 18 and 65 |
| `department` | String | One of: Engineering, Sales, HR, Finance |
| `salary` | Float | Must be positive |
| `join_date` | String | Format: YYYY-MM-DD |
| `email` | String | Must contain @ |
| `active` | String | Must be 'yes' or 'no' |

This definition — what each column should look like — is called a **schema**. We will use it as our reference throughout the session.

In [ ]:
# Here is our raw employee data.
# Look carefully at the values — can you already spot anything odd?

raw_data = """\
employee_id,name,age,department,salary,join_date,email,active
101,Ananya Sharma,29,Engineering,72000.00,2021-03-15,ananya@company.com,yes
102,Rohan Mehta,,Sales,55000.00,2020-07-01,rohan@company.com,yes
103,Priya Nair,twenty-two,HR,48000.00,2019-11-20,priyanair@company.com,no
104,Vikram Rao,45,Finance,91000.00,15/08/2018,vikram@company.com,YES
105,Sneha Pillai,30,Engineering,68000.00,2022-01-10,sneha_at_company.com,yes
106,Arjun Das,72,Sales,53000.00,2021-06-25,arjun@company.com,yes
107,,28,HR,,2020-09-30,kavya@company.com,no
108,Deepak Joshi,35,Marketing,61000.00,2018-04-12,deepak@company.com,yes
109,Meena Iyer,31,finance,74000.00,2023-02-28,meena@company.com,yes
110,Kiran Bose,27,Engineering,-5000.00,2022-08-05,kiran@company.com,yes
"""

# Parse the raw CSV string into a list of dictionaries
reader = csv.DictReader(StringIO(raw_data))
employees = list(reader)

# Print the first two records to see the structure
for emp in employees[:2]:
    print(emp)
    print()

{'employee_id': '101', 'name': 'Ananya Sharma', 'age': '29', 'department': 'Engineering', 'salary': '72000.00', 'join_date': '2021-03-15', 'email': 'ananya@company.com', 'active': 'yes'}

{'employee_id': '102', 'name': 'Rohan Mehta', 'age': '', 'department': 'Sales', 'salary': '55000.00', 'join_date': '2020-07-01', 'email': 'rohan@company.com', 'active': 'yes'}



### Whiteboard discussion

Before we write any validation code, let's do a visual scan together.

Look at the raw data above and try to find every problem. We'll note them on the whiteboard.

Things to ask yourself:
- Are there any blank cells?
- Do all the numbers look like numbers?
- Do all the dates look the same?
- Do all the emails look valid?
- Are there any values that seem impossible?

In [ ]:
# Let's print all records in a readable format
print(f"Total records: {len(employees)}")
print("---")

for emp in employees:
    print(f"  ID {emp['employee_id']:>4} | {emp['name']:<18} | age: {emp['age']:<12} | dept: {emp['department']:<12} | salary: {emp['salary']:<10} | date: {emp['join_date']:<12} | email: {emp['email']:<28} | active: {emp['active']}")

Total records: 10
---
  ID  101 | Ananya Sharma      | age: 29           | dept: Engineering  | salary: 72000.00   | date: 2021-03-15   | email: ananya@company.com           | active: yes
  ID  102 | Rohan Mehta        | age:              | dept: Sales        | salary: 55000.00   | date: 2020-07-01   | email: rohan@company.com            | active: yes
  ID  103 | Priya Nair         | age: twenty-two   | dept: HR           | salary: 48000.00   | date: 2019-11-20   | email: priyanair@company.com        | active: no
  ID  104 | Vikram Rao         | age: 45           | dept: Finance      | salary: 91000.00   | date: 15/08/2018   | email: vikram@company.com           | active: YES
  ID  105 | Sneha Pillai       | age: 30           | dept: Engineering  | salary: 68000.00   | date: 2022-01-10   | email: sneha_at_company.com         | active: yes
  ID  106 | Arjun Das          | age: 72           | dept: Sales        | salary: 53000.00   | date: 2021-06-25   | email: arjun@company.com         

---
## Part 2 — Handling Missing Values

### What counts as "missing"?

When you read data from a CSV file in Python, missing values are usually just **empty strings** — the value is `""` (nothing between two commas).

Other common representations:
- `""` — empty string (most common in CSV)
- `None` — Python's null value
- `"N/A"`, `"null"`, `"NULL"`, `"nan"` — text placeholders used by different systems

The first job of data validation is to **find all missing values** before they cause silent errors.

In [ ]:
# Step 1: Find all missing values across the entire dataset

print("Scanning for missing values...")
print("=" * 45)

missing_found = False

for emp in employees:
    for field, value in emp.items():
        if value == "" or value is None:
            print(f"  MISSING  |  employee_id={emp['employee_id']}  |  field='{field}'")
            missing_found = True

if not missing_found:
    print("  No missing values found.")

print("=" * 45)

Scanning for missing values...
  MISSING  |  employee_id=102  |  field='age'
  MISSING  |  employee_id=107  |  field='name'
  MISSING  |  employee_id=107  |  field='salary'


In [ ]:
# how many emp do not have the salary data?

emp_without_sal = 0

for emp in employees:
  if emp['salary'] == "" or emp['salary'] is None:
    emp_without_sal = emp_without_sal + 1

print(f"The number of emp without salary data = {emp_without_sal}")

The number of emp without salary data = 1


### What do we do about missing values?

There is no single right answer. It depends on context. Here are the main options:

| Strategy | When to use it | Example |
|---|---|---|
| **Flag it** | Always a good first step | Print a warning |
| **Skip the record** | When the missing field is required | Don't process the row |
| **Fill with a default** | When a sensible default exists | `active` defaults to `"no"` |
| **Fill with an average** | For numeric fields (taught later with Pandas) | Fill `age` with mean age |

For today, we will use a combination: **flag critical problems** and **fill with a safe default** where possible.

In [ ]:
# Step 2: Define which fields are required vs. optional with defaults

# Required fields — if these are missing, the record is invalid
REQUIRED_FIELDS = ["employee_id", "name", "department", "salary"]

# Optional fields — if missing, we can fill with a default
DEFAULTS = {
    "active": "no",
    "age": "unknown"
}

valid_records = []
invalid_records = []

for emp in employees:
    # Check required fields
    is_valid = True
    for field in REQUIRED_FIELDS:
        if emp[field] == "" or emp[field] is None:
            print(f"INVALID RECORD: employee_id={emp['employee_id']} is missing required field '{field}'")
            is_valid = False

    if is_valid:
        # Fill in defaults for optional missing fields
        for field, default in DEFAULTS.items():
            if emp[field] == "" or emp[field] is None:
                print(f"  Filling default for employee_id={emp['employee_id']}: {field} = '{default}'")
                emp[field] = default
        valid_records.append(emp)
    else:
        invalid_records.append(emp)

print()
print(f"Valid records:   {len(valid_records)}")
print(f"Invalid records: {len(invalid_records)}")

  Filling default for employee_id=102: age = 'unknown'
INVALID RECORD: employee_id=107 is missing required field 'name'
INVALID RECORD: employee_id=107 is missing required field 'salary'

Valid records:   9
Invalid records: 1


---
### Exercise 1 — Find Missing Values

**Task:** A new batch of employee data has arrived. Your job is to count how many records have at least one missing field.

Complete the function below. It should return the **count** of records that have one or more missing values in any field.

In [ ]:
new_batch = [
    {"employee_id": "201", "name": "Rahul Sen",      "age": "33", "department": "Sales"},
    {"employee_id": "202", "name": "",               "age": "28", "department": "HR"},
    {"employee_id": "203", "name": "Divya Menon",    "age": "",   "department": "Engineering"},
    {"employee_id": "204", "name": "Suresh Kumar",   "age": "41", "department": ""},
    {"employee_id": "205", "name": "Lakshmi Reddy",  "age": "25", "department": "Finance"},
]

def count_records_with_missing(records):
    """
    Returns the number of records that have at least one empty field.
    """
    count = 0
    for record in records:
        for field, value in record.items():
            if value == ___:          # <-- what value means "missing" in a CSV?
                count += ___          # <-- increment the count
                break                 # only count this record once even if multiple fields are missing
    return count


result = count_records_with_missing(new_batch)
print(f"Records with at least one missing field: {result}")
print(f"Expected: 3")
print(f"Correct: {result == 3}")

<details>
<summary>Show solution</summary>

```python
def count_records_with_missing(records):
    count = 0
    for record in records:
        for field, value in record.items():
            if value == "":
                count += 1
                break
    return count
```

</details>

---
## Part 3 — Fixing Wrong Data Types

### The fundamental problem

When Python reads a CSV file, **everything comes in as a string** — even numbers.

This means that after reading:
- `age` is `"29"` — not the number `29`
- `salary` is `"72000.00"` — not the float `72000.0`

You cannot do math on strings. Try this:

In [ ]:
"29" + "28"

'2928'

In [ ]:
# Demonstration: what happens when you treat strings as numbers

age_as_string = "29"
age_as_number = 29

print("String + string:",  age_as_string + age_as_string)   # What do you expect?
print("Number + number:",  age_as_number + age_as_number)   # What do you expect?

print()
print("Type of string version:", type(age_as_string))
print("Type of number version:", type(age_as_number))

### Converting types in Python

Python gives us built-in functions to convert between types:

| Function | Does what | Example |
|---|---|---|
| `int(x)` | Converts `x` to an integer | `int("29")` → `29` |
| `float(x)` | Converts `x` to a decimal number | `float("72000.00")` → `72000.0` |
| `str(x)` | Converts `x` to a string | `str(29)` → `"29"` |

But what if the conversion is **impossible**? Like trying `int("twenty-two")`? Python will raise an error. We need to handle that.

In [ ]:
int('29')

29

In [ ]:
int('twenty nine')

ValueError: invalid literal for int() with base 10: 'twenty nine'

In [ ]:
# A safe conversion function using try/except

def safe_int(value, fallback=None):
    """
    Tries to convert value to an integer.
    If it fails, returns fallback instead of crashing.
    """
    try:
        return int(value)
    except (ValueError, TypeError):
        return fallback

def safe_float(value, fallback=None):
    """
    Tries to convert value to a float.
    If it fails, returns fallback instead of crashing.
    """
    try:
        return float(value)
    except (ValueError, TypeError):
        return fallback


# Test these functions
test_values = ["29", "twenty-two", "", "45.5", "0"]

print("Testing safe_int:")
for v in test_values:
    result = safe_int(v)
    print(f"  safe_int({v!r:>14}) = {str(result):>8}  (type: {type(result).__name__})")

Testing safe_int:
  safe_int(          '29') =       29  (type: int)
  safe_int(  'twenty-two') =     None  (type: NoneType)
  safe_int(            '') =     None  (type: NoneType)
  safe_int(        '45.5') =     None  (type: NoneType)
  safe_int(           '0') =        0  (type: int)


In [ ]:
safe_int('asdasdasd', 20)

20

In [ ]:
# Now apply type validation to our employee dataset

print("Checking data types for each record...")
print("=" * 55)

type_issues = []  # We will collect all issues here

for emp in valid_records:
    eid = emp["employee_id"]

    # Check age
    age_converted = safe_int(emp["age"])
    if emp["age"] not in ("", "unknown") and age_converted is None:
        issue = f"  employee_id={eid} | age='{emp['age']}' cannot be converted to integer"
        print(issue)
        type_issues.append({"id": eid, "field": "age", "raw_value": emp["age"]})
    else:
      emp['age'] = age_converted

    # Check salary
    salary_converted = safe_float(emp["salary"])
    if emp["salary"] != "" and salary_converted is None:
        issue = f"  employee_id={eid} | salary='{emp['salary']}' cannot be converted to float"
        print(issue)
        type_issues.append({"id": eid, "field": "salary", "raw_value": emp["salary"]})

print()
if type_issues:
    print(f"Found {len(type_issues)} type conversion issue(s).")
else:
    print("All type checks passed.")

Checking data types for each record...
  employee_id=103 | age='twenty-two' cannot be converted to integer

Found 1 type conversion issue(s).


---
### Exercise 2 — Type Conversion

**Task:** Complete the `check_types` function below. It should go through a list of records and return a list of problems found, where each problem is a dictionary with the keys `id`, `field`, and `raw_value`.

Fields to check:
- `employee_id` must convert to `int`
- `age` must convert to `int` (skip if value is `""` or `"unknown"`)
- `salary` must convert to `float` (skip if value is `""`)

In [ ]:
test_records = [
    {"employee_id": "301", "name": "Arun Nair",    "age": "34",         "salary": "65000.00"},
    {"employee_id": "3O2", "name": "Bhavna Rao",   "age": "twenty-nine","salary": "58000.00"},
    {"employee_id": "303", "name": "Chetan Singh", "age": "27",         "salary": "abc"},
    {"employee_id": "304", "name": "Disha Patel",  "age": "",           "salary": "71000.00"},
]

def check_types(records):
    """
    Returns a list of type issues found.
    Each issue is a dict: {"id": ..., "field": ..., "raw_value": ...}
    """
    problems = []

    for record in records:
        eid = record["employee_id"]

        # Check employee_id — must be convertible to int
        if safe_int(eid) is ___:     # <-- what does safe_int return when conversion fails?
            problems.append({"id": eid, "field": "employee_id", "raw_value": eid})

        # Check age — skip if blank or unknown
        age_val = record["age"]
        if age_val not in ("", "unknown"):
            if safe_int(___) is None:    # <-- which variable holds the age value?
                problems.append({"id": eid, "field": "age", "raw_value": age_val})

        # Check salary — skip if blank
        salary_val = record["salary"]
        if salary_val != "":
            if safe_float(salary_val) is ___:    # <-- same pattern as above
                problems.append({"id": eid, "field": "salary", "raw_value": salary_val})

    return problems


found = check_types(test_records)
print(f"Problems found: {len(found)}")
for p in found:
    print(f"  {p}")
print()
print(f"Expected: 3 problems")
print(f"Correct: {len(found) == 3}")

<details>
<summary>Show solution</summary>

```python
def check_types(records):
    problems = []
    for record in records:
        eid = record["employee_id"]
        if safe_int(eid) is None:
            problems.append({"id": eid, "field": "employee_id", "raw_value": eid})
        age_val = record["age"]
        if age_val not in ("", "unknown"):
            if safe_int(age_val) is None:
                problems.append({"id": eid, "field": "age", "raw_value": age_val})
        salary_val = record["salary"]
        if salary_val != "":
            if safe_float(salary_val) is None:
                problems.append({"id": eid, "field": "salary", "raw_value": salary_val})
    return problems
```

</details>

---
## Part 4 — Identifying Inconsistent Formats

### Why format consistency matters

Even when a value is the correct type, it might be written in an inconsistent **format** compared to other records. This causes problems when you try to:
- Sort or compare values (`"YES"` vs `"yes"` — are they the same?)
- Parse dates (`"15/08/2018"` vs `"2018-08-15"` — both are valid dates, but a program expecting one format will reject the other)
- Group data (`"finance"` vs `"Finance"` — a group-by would split them into two separate groups)

We have three format inconsistencies in our dataset. Let's find and fix each one.

### Format Issue 1 — Dates

Our expected date format is `YYYY-MM-DD` (e.g. `2021-03-15`).  
But one record has the date as `15/08/2018` — the day and year are swapped, and slashes are used instead of dashes.

We need to detect this and flag it.

In [ ]:
# Checking date format consistency
# Expected format: YYYY-MM-DD
# We can check this by looking at the structure: 4 chars, dash, 2 chars, dash, 2 chars

def is_valid_date_format(date_str):
    """
    Returns True if the date string matches YYYY-MM-DD format.
    This is a simple structural check — not a full date validity check.
    """
    if len(date_str) != 10:
        return False
    if date_str[4] != "-" or date_str[7] != "-":
        return False
    year_part  = date_str[0:4]
    month_part = date_str[5:7]
    day_part   = date_str[8:10]
    if not (year_part.isdigit() and month_part.isdigit() and day_part.isdigit()):
        return False
    return True


# Test it on a few examples
test_dates = ["2021-03-15", "15/08/2018", "2023-13-01", "2022-01-10", ""]
print("Date format check:")
for d in test_dates:
    print(f"  '{d}'  ->  valid format: {is_valid_date_format(d)}")

print()
print("Scanning employee records...")
print("=" * 45)
for emp in valid_records:
    if not is_valid_date_format(emp["join_date"]):
        print(f"  BAD DATE FORMAT | employee_id={emp['employee_id']} | join_date='{emp['join_date']}'")

### Format Issue 2 — Inconsistent casing

Two types of casing problems in our data:

1. The `active` field has `"YES"` (uppercase) mixed with `"yes"` (lowercase)
2. The `department` field has `"finance"` (lowercase) mixed with `"Finance"` (title case)

The fix for casing is simple: standardize everything to one format.

In [ ]:
# Check the range of values in the 'active' and 'department' fields

active_values = set()
department_values = set()

for emp in valid_records:
    active_values.add(emp["active"])
    department_values.add(emp["department"])

print("Unique values in 'active'     :", active_values)
print("Unique values in 'department' :", department_values)

print()
print("--- Normalizing 'active' to lowercase ---")
for emp in valid_records:
    original = emp["active"]
    emp["active"] = emp["active"].lower().strip()
    if original != emp["active"]:
        print(f"  employee_id={emp['employee_id']} | '{original}' -> '{emp['active']}'")

print()
print("--- Normalizing 'department' to title case ---")
for emp in valid_records:
    original = emp["department"]
    emp["department"] = emp["department"].strip().title()
    if original != emp["department"]:
        print(f"  employee_id={emp['employee_id']} | '{original}' -> '{emp['department']}'")

print()
print("After normalization:")
print("Unique values in 'active'     :", set(emp["active"] for emp in valid_records))
print("Unique values in 'department' :", set(emp["department"] for emp in valid_records))

Unique values in 'active'     : {'YES', 'no', 'yes'}
Unique values in 'department' : {'Engineering', 'finance', 'Finance', 'Sales', 'HR', 'Marketing'}

--- Normalizing 'active' to lowercase ---
  employee_id=104 | 'YES' -> 'yes'

--- Normalizing 'department' to title case ---
  employee_id=103 | 'HR' -> 'Hr'
  employee_id=109 | 'finance' -> 'Finance'

After normalization:
Unique values in 'active'     : {'no', 'yes'}
Unique values in 'department' : {'Engineering', 'Finance', 'Hr', 'Sales', 'Marketing'}


---
### Exercise 3 — Spot the Format Issues

**Task:** The function below should detect format problems in a list of records. Fill in the blanks to complete the two checks:

1. Flag any `join_date` that is not in `YYYY-MM-DD` format
2. Flag any `active` value that is not exactly `"yes"` or `"no"` (after lowercasing)

In [ ]:
sample_records = [
    {"employee_id": "401", "join_date": "2022-04-15", "active": "yes"},
    {"employee_id": "402", "join_date": "20/03/2021", "active": "YES"},
    {"employee_id": "403", "join_date": "2023-11-01", "active": "maybe"},
    {"employee_id": "404", "join_date": "2021-07-30", "active": "No"},
    {"employee_id": "405", "join_date": "2024-01-12", "active": "yes"},
]

VALID_ACTIVE_VALUES = ["yes", "no"]

def find_format_issues(records):
    """
    Returns a list of format issues.
    Each issue is a dict: {"id": ..., "field": ..., "value": ..., "problem": ...}
    """
    issues = []
    for record in records:
        eid = record["employee_id"]

        # Check 1: join_date format
        if not is_valid_date_format(record[___]):    # <-- which field are we checking?
            issues.append({
                "id": eid,
                "field": "join_date",
                "value": record["join_date"],
                "problem": "not in YYYY-MM-DD format"
            })

        # Check 2: active must be 'yes' or 'no' (after lowercasing)
        active_normalized = record["active"].lower().strip()
        if active_normalized not in ___:    # <-- which list do we check against?
            issues.append({
                "id": eid,
                "field": "active",
                "value": record["active"],
                "problem": f"unexpected value '{record['active']}'"
            })

    return issues


issues = find_format_issues(sample_records)
print(f"Format issues found: {len(issues)}")
for issue in issues:
    print(f"  {issue}")
print()
print(f"Expected: 2 issues  (record 402 has bad date, record 403 has bad active value)")
print(f"Note: 402's active='YES' and 404's active='No' should NOT be flagged — they normalize to valid values")
print(f"Correct: {len(issues) == 2}")

<details>
<summary>Show solution</summary>

```python
def find_format_issues(records):
    issues = []
    for record in records:
        eid = record["employee_id"]
        if not is_valid_date_format(record["join_date"]):
            issues.append({
                "id": eid, "field": "join_date",
                "value": record["join_date"], "problem": "not in YYYY-MM-DD format"
            })
        active_normalized = record["active"].lower().strip()
        if active_normalized not in VALID_ACTIVE_VALUES:
            issues.append({
                "id": eid, "field": "active",
                "value": record["active"], "problem": f"unexpected value '{record['active']}'"
            })
    return issues
```

</details>

---
## Part 5 — Schema Validation Basics

### What is a schema?

A **schema** is a formal description of what your data should look like. It answers questions like:
- What columns exist?
- What type should each column be?
- Are there allowed values for a column?
- What are the minimum and maximum allowed values?

We already saw our schema as a table at the top of the notebook. Now we are going to represent it **in code** and use it to automatically validate every record.

---

### The key idea: separating rules from logic

Instead of writing separate `if` statements for each field, we define all the rules in one place as a data structure. Then one piece of code reads those rules and applies them to every record.

This is a much more maintainable approach. If the rules change, you update the schema — you do not have to hunt through all your validation code.

In [ ]:
# Define the schema as a list of rules.
# Each rule is a dictionary describing what one field must look like.

EMPLOYEE_SCHEMA = [
    {
        "field":    "employee_id",
        "required": True,
        "type":     "int"
    },
    {
        "field":    "name",
        "required": True,
        "type":     "str"
    },
    {
        "field":    "age",
        "required": False,
        "type":     "int",
        "min":      18,
        "max":      65
    },
    {
        "field":    "department",
        "required": True,
        "type":     "str",
        "allowed":  ["Engineering", "Sales", "Hr", "Finance"]
    },
    {
        "field":    "salary",
        "required": True,
        "type":     "float",
        "min":      0.0
    },
    {
        "field":    "join_date",
        "required": True,
        "type":     "str"
    },
    {
        "field":    "email",
        "required": True,
        "type":     "str",
        "must_contain": "@"
    },
    {
        "field":    "active",
        "required": True,
        "type":     "str",
        "allowed":  ["yes", "no"]
    }
]

print(f"Schema defined with {len(EMPLOYEE_SCHEMA)} field rules.")
print()
for rule in EMPLOYEE_SCHEMA:
    extras = []
    if "min"          in rule: extras.append(f"min={rule['min']}")
    if "max"          in rule: extras.append(f"max={rule['max']}")
    if "allowed"      in rule: extras.append(f"allowed={rule['allowed']}")
    if "must_contain" in rule: extras.append(f"must_contain='{rule['must_contain']}'")
    extra_str = "  |  " + ", ".join(extras) if extras else ""
    print(f"  {rule['field']:<15} type={rule['type']:<6}  required={str(rule['required']):<5}{extra_str}")

Schema defined with 8 field rules.

  employee_id     type=int     required=True 
  name            type=str     required=True 
  age             type=int     required=False  |  min=18, max=65
  department      type=str     required=True   |  allowed=['Engineering', 'Sales', 'Hr', 'Finance']
  salary          type=float   required=True   |  min=0.0
  join_date       type=str     required=True 
  email           type=str     required=True   |  must_contain='@'
  active          type=str     required=True   |  allowed=['yes', 'no']


In [ ]:
# The schema validator function
# This reads the schema rules and checks each record against every rule.

def validate_record(record, schema):
    """
    Validates a single record against a schema.
    Returns a list of violation messages (empty list means the record is valid).
    """
    violations = []

    for rule in schema:
        field   = rule["field"]
        value   = record.get(field, "")  # get field value, default to empty string if field is absent

        # Rule 1: Required check
        if rule.get("required") and (value == "" or value is None):
            violations.append(f"'{field}' is required but missing")
            continue  # skip further checks for this field

        if value == "" or value is None:
            continue  # field is optional and empty — skip

        # Rule 2: Type check
        field_type = rule.get("type")
        converted_value = value  # we'll update this if conversion succeeds

        if field_type == "int":
            converted_value = safe_int(value)
            if converted_value is None:
                violations.append(f"'{field}' should be an integer, got '{value}'")
                continue
        elif field_type == "float":
            converted_value = safe_float(value)
            if converted_value is None:
                violations.append(f"'{field}' should be a number, got '{value}'")
                continue

        # Rule 3: Min / Max check (only applies after successful type conversion)
        if "min" in rule and converted_value is not None:
            if converted_value < rule["min"]:
                violations.append(f"'{field}' value {converted_value} is below minimum {rule['min']}")

        if "max" in rule and converted_value is not None:
            if converted_value > rule["max"]:
                violations.append(f"'{field}' value {converted_value} exceeds maximum {rule['max']}")

        # Rule 4: Allowed values check
        if "allowed" in rule:
            if str(value).strip().lower() not in [a.lower() for a in rule["allowed"]]:
                violations.append(f"'{field}' value '{value}' not in allowed list {rule['allowed']}")

        # Rule 5: Must contain check
        if "must_contain" in rule:
            if rule["must_contain"] not in str(value):
                violations.append(f"'{field}' value '{value}' must contain '{rule['must_contain']}'")

    return violations


print("Schema validator defined.")
print()
print("Quick test on a known bad record:")
bad_record = {
    "employee_id": "abc",
    "name": "Test Person",
    "age": "70",
    "department": "Accounts",
    "salary": "-100",
    "join_date": "2021-01-01",
    "email": "testperson_no_at_sign",
    "active": "yes"
}
test_violations = validate_record(bad_record, EMPLOYEE_SCHEMA)
for v in test_violations:
    print(f"  VIOLATION: {v}")

In [ ]:
# Run the full schema validation over all valid_records

print("Full schema validation report")
print("=" * 55)

total_violations = 0
clean_count = 0

for emp in valid_records:
    violations = validate_record(emp, EMPLOYEE_SCHEMA)
    if violations:
        print(f"\n  employee_id={emp['employee_id']} ({emp['name']})")
        for v in violations:
            print(f"    -> {v}")
        total_violations += len(violations)
    else:
        clean_count += 1

print()
print("=" * 55)
print(f"Clean records:      {clean_count}")
print(f"Records with issues:{len(valid_records) - clean_count}")
print(f"Total violations:   {total_violations}")

---
### Exercise 4 — Extend the Schema

**Task:** The company has introduced a new field called `"experience_years"` with the following rules:
- It is optional (not required)
- It must be an integer
- It must be between 0 and 40

Add this rule to the schema and re-run the validation on the sample records below.

In [ ]:
# Add the new rule to the schema here
NEW_RULE = {
    "field":    "experience_years",
    "required": ___,      # <-- is it required?
    "type":     ___,      # <-- what type should it be?
    "min":      ___,      # <-- minimum value
    "max":      ___       # <-- maximum value
}

EXTENDED_SCHEMA = EMPLOYEE_SCHEMA + [NEW_RULE]

# Test records
schema_test_records = [
    {"employee_id": "501", "name": "Arjun Bose",   "age": "30", "department": "Engineering",
     "salary": "70000", "join_date": "2020-01-01", "email": "arjun@company.com",
     "active": "yes", "experience_years": "5"},

    {"employee_id": "502", "name": "Farida Khan",  "age": "28", "department": "Sales",
     "salary": "62000", "join_date": "2021-06-15", "email": "farida@company.com",
     "active": "yes", "experience_years": "45"},   # <-- experience out of range

    {"employee_id": "503", "name": "Gaurav Tiwari","age": "35", "department": "Finance",
     "salary": "85000", "join_date": "2019-03-20", "email": "gaurav@company.com",
     "active": "no",  "experience_years": ""},     # <-- optional, so blank is fine
]

print("Validation with extended schema:")
for rec in schema_test_records:
    violations = validate_record(rec, EXTENDED_SCHEMA)
    if violations:
        print(f"  employee_id={rec['employee_id']}: {violations}")
    else:
        print(f"  employee_id={rec['employee_id']}: CLEAN")

print()
print("Expected: 501=CLEAN, 502 has a violation, 503=CLEAN")

<details>
<summary>Show solution</summary>

```python
NEW_RULE = {
    "field":    "experience_years",
    "required": False,
    "type":     "int",
    "min":      0,
    "max":      40
}
```

</details>

---
## Part 6 — Putting It All Together: A Validation Pipeline

We have built several individual tools:
- Missing value detection
- Type conversion with safe fallbacks
- Format consistency checks
- Schema-driven validation

In a real project, you would combine all of these into a single **pipeline** — a function that takes raw data and returns a clean report.

Let's build that now.

In [ ]:
def run_validation_pipeline(raw_csv_string, schema):
    """
    Runs the full data validation pipeline on a raw CSV string.

    Steps:
      1. Parse the CSV into records
      2. Detect missing values in required fields
      3. Normalise casing for 'active' and 'department'
      4. Run schema validation on every record
      5. Return a summary report
    """
    reader  = csv.DictReader(StringIO(raw_csv_string))
    records = list(reader)

    report = {
        "total":            len(records),
        "clean":            0,
        "with_issues":      0,
        "issue_details":    []
    }

    for rec in records:
        # Step: normalise casing
        if "active" in rec:
            rec["active"] = rec["active"].lower().strip()
        if "department" in rec:
            rec["department"] = rec["department"].strip().title()

        # Step: schema validation
        violations = validate_record(rec, schema)

        if violations:
            report["with_issues"] += 1
            report["issue_details"].append({
                "employee_id": rec.get("employee_id", "?"),
                "name":        rec.get("name", "?"),
                "violations":  violations
            })
        else:
            report["clean"] += 1

    return report


# Run the pipeline on our original dataset
report = run_validation_pipeline(raw_data, EMPLOYEE_SCHEMA)

print("DATA VALIDATION REPORT")
print("=" * 55)
print(f"Total records:        {report['total']}")
print(f"Clean records:        {report['clean']}")
print(f"Records with issues:  {report['with_issues']}")
print()
print("Issues found:")
for entry in report["issue_details"]:
    print(f"\n  [{entry['employee_id']}] {entry['name']}")
    for v in entry["violations"]:
        print(f"      -> {v}")

---
## Final Exercise — End-to-End Challenge

This is the session's capstone exercise. You will work with a brand new dataset that has a mix of all the issues we covered today.

**Your task:** Use `run_validation_pipeline` with the `EMPLOYEE_SCHEMA` to validate the data below. Then answer the three questions in the cell after it.

In [ ]:
challenge_data = """\
employee_id,name,age,department,salary,join_date,email,active
601,Nisha Gupta,32,HR,54000.00,2020-05-10,nisha@company.com,yes
602,Tarun Kapoor,,Sales,49000.00,2021-09-14,tarun@company.com,YES
603,Usha Reddy,nineteen,Finance,38000.00,2022-03-01,usha@company.com,no
604,,40,Engineering,80000.00,2019-07-22,hari@company.com,yes
605,Vinod Pillai,45,Operations,61000.00,2018/12/05,vinod@company.com,no
606,Wasim Khan,28,Sales,-200.00,2023-01-15,wasim_company.com,yes
607,Zara Thomas,35,Finance,77000.00,2021-04-30,zara@company.com,no
"""

# Run the pipeline
challenge_report = run_validation_pipeline(challenge_data, EMPLOYEE_SCHEMA)

print("CHALLENGE VALIDATION REPORT")
print("=" * 55)
print(f"Total records:        {challenge_report['total']}")
print(f"Clean records:        {challenge_report['clean']}")
print(f"Records with issues:  {challenge_report['with_issues']}")
print()
for entry in challenge_report["issue_details"]:
    print(f"  [{entry['employee_id']}] {entry['name']}")
    for v in entry["violations"]:
        print(f"      -> {v}")

In [ ]:
# Answer these questions based on the output above.
# Replace None with your answers.

# Q1: How many records are completely clean (no violations)?
answer_clean = None

# Q2: Which employee_id has a negative salary?
answer_negative_salary_id = None

# Q3: Which employee_id has a department that is not in the allowed list?
answer_bad_department_id = None


# Grading
print("Checking your answers...")
print(f"Q1 (clean count):          {'CORRECT' if answer_clean == 2 else 'Try again'}")
print(f"Q2 (negative salary id):   {'CORRECT' if str(answer_negative_salary_id) == '606' else 'Try again'}")
print(f"Q3 (bad department id):    {'CORRECT' if str(answer_bad_department_id) == '605' else 'Try again'}")

---
## Session Summary

Here is a recap of everything we covered today:

### 1. Missing Values
- CSV files represent missing values as empty strings `""`
- Distinguish between **required** fields (must be present) and **optional** fields (can be filled with a default)
- Always scan for missing values **before** doing any computation

### 2. Wrong Data Types
- Everything from a CSV comes in as a **string**
- Use `int()` and `float()` to convert — but always wrap in `try/except` to handle invalid values safely
- A helper function like `safe_int()` makes your code more robust and reusable

### 3. Inconsistent Formats
- Casing issues: standardise with `.lower()` or `.title()`
- Date format issues: check structure manually or use Python's `datetime` library (covered in a later session)
- Always decide on a **canonical format** first, then normalise everything to it

### 4. Schema Validation
- A schema is a set of rules that defines what your data should look like
- Representing the schema as a data structure (a list of dictionaries) separates the **rules** from the **logic**
- A generic validator can then apply any schema to any dataset

---

### What comes next

In the coming modules, we will use **Pandas** — a powerful library that has all of these validation and cleaning tools built in, working on thousands of rows at once. The concepts you learned today are exactly what Pandas is doing under the hood.

---

### Key Python concepts used today

| Concept | Used for |
|---|---|
| `try / except` | Safe type conversion |
| `set()` | Finding unique values in a column |
| `.lower()`, `.strip()`, `.title()` | Normalising string formats |
| List of dictionaries | Representing tabular data and schema rules |
| `csv.DictReader` | Reading CSV rows as dictionaries |